# Data Formatting
## customer_id / postal_code の変換テーブル作成
長い hex 文字列を連番整数に変換する。

In [1]:
%load_ext cudf.pandas


ModuleNotFoundError: No module named 'cudf'

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
print(f"rows: {len(customers):,}  unique customer_id: {customers['customer_id'].nunique():,}")
customers.head(3)

rows: 1,371,980  unique customer_id: 1,371,980


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...


In [4]:
# customer_id 変換テーブル
unique_ids = customers["customer_id"].unique()
id_map = pd.DataFrame({
    "customer_id": unique_ids,
    "customer_id_int": range(1, len(unique_ids) + 1)
})

id_map.to_csv(PROCESSED_DIR / "customer_id_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'customer_id_map.csv'}")
id_map.head(3)

変換テーブル保存完了: ../../data/processed/customer_id_map.csv


,customer_id,customer_id_int
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,1
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,3


In [5]:
# postal_code 変換テーブル
unique_postal = customers["postal_code"].unique()
postal_map = pd.DataFrame({
    "postal_code": unique_postal,
    "postal_code_int": range(1, len(unique_postal) + 1)
})

postal_map.to_csv(PROCESSED_DIR / "postal_code_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'postal_code_map.csv'}")
print(f"unique postal_code: {len(postal_map):,}")
postal_map.head(3)

変換テーブル保存完了: ../../data/processed/postal_code_map.csv
unique postal_code: 352,899


,postal_code,postal_code_int
0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,1
1,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,2
2,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,3


In [6]:
# customer_id と postal_code を両方整数に変換して保存
customers_converted = (
    customers
    .merge(id_map, on="customer_id", how="left")
    .merge(postal_map, on="postal_code", how="left")
    .drop(columns=["customer_id", "postal_code"])
    .rename(columns={"customer_id_int": "customer_id", "postal_code_int": "postal_code"})
)
cols = ["customer_id", "FN", "Active", "club_member_status", "fashion_news_frequency", "age", "postal_code"]
customers_converted = customers_converted[cols]

customers_converted.to_csv(PROCESSED_DIR / "customers_converted.csv", index=False)
print(f"変換済み customers 保存完了: {PROCESSED_DIR / 'customers_converted.csv'}")
customers_converted.head(3)

変換済み customers 保存完了: ../../data/processed/customers_converted.csv


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,1,NaN,NaN,ACTIVE,NONE,49.0,1
1,2,NaN,NaN,ACTIVE,NONE,25.0,2
2,3,NaN,NaN,ACTIVE,NONE,24.0,3


In [ ]:
# transactions_train の customer_id を変換テーブルで整数に変換して保存
transactions = pd.read_csv(
    RAW_DIR / "transactions_train.csv",
    dtype={"article_id": str}
)
print(f"rows: {len(transactions):,}")

transactions_converted = (
    transactions
    .merge(id_map, on="customer_id", how="left")
    .drop(columns=["customer_id"])
    .rename(columns={"customer_id_int": "customer_id"})
)
cols = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
transactions_converted = transactions_converted[cols]

transactions_converted.to_csv(PROCESSED_DIR / "transactions_converted.csv", index=False)
print(f"変換済み transactions 保存完了: {PROCESSED_DIR / 'transactions_converted.csv'}")
transactions_converted.head(3)

rows: 31,788,324
変換済み transactions 保存完了: ../../data/processed/transactions_converted.csv


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,3,0663713001,0.050831,2
1,2018-09-20,3,0541518023,0.030492,2
2,2018-09-20,8,0505221004,0.015237,2


: 

In [3]:
articles = pd.read_csv(RAW_DIR / "articles.csv")
print(f"rows: {len(articles):,}  unique index_name: {customers['index_name'].nunique():,}")
articles.head(3)

rows: 105,542  unique index_name: 10


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [1]:
# index_nameとindex_group_nameの組み合わせ
combo = articles.groupby(['index_group_name', 'index_name']).size().reset_index(name='count')
print(combo.to_string())

NameError: name 'articles' is not defined

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

# 必要なカラムだけ読み込む
articles = pd.read_csv(DATA_DIR / 'raw/articles.csv', usecols=['article_id', 'index_group_name'])
transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', usecols=['article_id', 'week'])

# 最新weekから4週間分を抽出
max_week = transactions['week'].max()
recent = transactions[transactions['week'] >= max_week - 3]

# articlesと結合
recent = recent.merge(articles, on='article_id', how='left')

# index_group_nameごとの取引回数
result = recent.groupby('index_group_name').size().reset_index(name='count').sort_values('count', ascending=False)
print(result.to_string())

  index_group_name   count
2       Ladieswear  620485
1          Divided  233676
3         Menswear   61546
4            Sport   57400
0    Baby/Children   24256


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = (Path.cwd() / '../../data').resolve()

articles = pd.read_csv(DATA_DIR / 'raw/articles.csv', usecols=['article_id', 'index_group_name'])
transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', usecols=['customer_id', 'article_id', 'week'])

# 最新4週間
max_week = transactions['week'].max()
recent = transactions[transactions['week'] >= max_week - 3]

# articlesと結合
recent = recent.merge(articles, on='article_id', how='left')

# ユーザーごとに購入したindex_group_nameの組み合わせを作成
user_groups = (
    recent.groupby('customer_id')['index_group_name']
    .apply(lambda x: ' + '.join(sorted(set(x))))
    .reset_index(name='combination')
)

# 組み合わせごとのユーザー数
result = user_groups.groupby('combination').size().reset_index(name='user_count').sort_values('user_count', ascending=False)
print(result.to_string())


                                                combination  user_count
24                                               Ladieswear       93794
17                                     Divided + Ladieswear       53925
16                                                  Divided       22863
28                                                 Menswear        7884
27                                       Ladieswear + Sport        7232
30                                                    Sport        7182
25                                    Ladieswear + Menswear        6212
20                             Divided + Ladieswear + Sport        5755
18                          Divided + Ladieswear + Menswear        5053
0                                             Baby/Children        2975
9                                Baby/Children + Ladieswear        2687
2                      Baby/Children + Divided + Ladieswear        2029
21                                       Divided + Menswear     